# 01 — Dataset acquisition and export

This notebook runs the existing `guesser-data` command-line workflow: **download → prepare → optional preview → export**. The implementation stays in `models/guesser/data/src/`; this notebook only supplies options and calls the public CLI.

> **Production caution:** downloading all Quick, Draw! categories reads the full source files and can take a long time. Do not enable the pipeline while another data command is running; let the active job finish rather than starting a second one here.

The data package uses Python 3.13 while this Jupyter workspace uses Python 3.12, so importing its implementation modules into this kernel would cross the project's environment boundary. Calling the existing CLI through `uv` reuses the exact implementation without copying its logic. The next code cell can run `uv sync --project models/guesser/data --frozen` from the repository root. Keep its opt-in disabled while any data command is active; syncing may update packages in the live environment.




In [ ]:
import json
import os
import subprocess
from pathlib import Path

from guesser_evaluation import data_environment
from guesser_evaluation.notebook_tools import module_source_path

# Resolve project paths from the installed workspace, independent of the notebook cwd.
EVALUATION_PROJECT = module_source_path("notebook_tools.py").parents[2]
REPOSITORY_ROOT = EVALUATION_PROJECT.parents[2]
DATA_PROJECT = EVALUATION_PROJECT.parent / "data"

# Dependency sync can change packages used by a live downloader, so require an opt-in.
SYNC_DATA_ENVIRONMENT = False
if SYNC_DATA_ENVIRONMENT:
    # Run the documented setup command from the repository root.
    subprocess.run(
        ["uv", "sync", "--project", "models/guesser/data", "--frozen"],
        cwd=REPOSITORY_ROOT,
        check=True,
    )
else:
    print("Environment sync skipped; set SYNC_DATA_ENVIRONMENT = True to run it.")

# Read configuration with the data project's interpreter; do not import across environments.
DEFAULTS_QUERY = """
import json
from guesser_data.cli import parser
from guesser_data.settings import DOWNLOAD_METHODS, PREPROCESS_DEFAULTS, PROFILES, TRAINABLE_PROFILES, Settings
defaults = parser().parse_args(["download"])
print(json.dumps({
    "profile": defaults.profile, "data_root": str(defaults.data_root),
    "concurrency": defaults.concurrency, "profiles": PROFILES,
    "trainable_profiles": TRAINABLE_PROFILES, "download_methods": DOWNLOAD_METHODS,
    "seed": Settings().seed, "preprocessing": PREPROCESS_DEFAULTS,
}))
"""
configuration_result = subprocess.run(
    [
        "uv",
        "run",
        "--project",
        str(DATA_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-c",
        DEFAULTS_QUERY,
    ],
    cwd=REPOSITORY_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
DATA_DEFAULTS = json.loads(configuration_result.stdout)
print("Data package defaults (read-only snapshot; rerun setup after source changes):")
print(json.dumps(DATA_DEFAULTS, indent=2))

## Options

Change the corresponding values in the next code cell. `None`/empty values leave defaults with the data package rather than recreating them here. Source defaults are in [`settings.py`](../../data/src/settings.py); exact command flags are shown by `guesser-data <command> --help`.

| Option | Notebook default | Choices / effect |
|---|---|---|
| `PROFILE` (`--profile`) | CLI default from `DATA_DEFAULTS` | Choose a `trainable_profiles` entry shown above: currently `production`, `medium`, or `small`. All require all 345 labels. `smoke` is example/probe only and cannot run this pipeline. |
| `DATA_ROOT` (`--data-root`) | `None` | Delegate to the CLI default shown above; otherwise provide the parent data directory (the profile subdirectory is added by the CLI). Relative paths resolve from the repository root. |
| `CONCURRENCY` (`--concurrency`) | `None` | Delegate to the CLI default shown above; supported overrides are 1, 2, or 3. |
| `CANDIDATE_RESERVE` (`--candidates`) | `None` | Delegate to the selected profile's reserve shown above. An override may enlarge, but not reduce, that reserve. |
| `CATEGORY_FILTER` (`--class`) | empty tuple | Download all categories, or name specific categories to resume/enlarge. This filters download only: prepare still requires all 345 categories for a trainable profile. |
| `EXPORT_OUTPUT_DIR` (`--output-dir`) | `None` | Use the CLI's profile-local `exports/` directory, or override the archive directory. Relative paths resolve from the repository root. |
| `INCLUDE_PREVIEW` | `False` | If enabled, create the optional training-only preview between prepare and export. |
| `RUN_PIPELINE` | `False` | Must be set to `True` explicitly before the production steps execute. Run the full sequence only when no other data process is active. |

The setup cell reads profile counts, sampling methods, seed, preprocessing settings, and CLI defaults from the actual data package, rather than copying their values here. Production reads complete source files; medium/small use prefix samples, which save transfer but may be less representative. Profiles use separate directories and held-out originals, so they are not a controlled dataset-size comparison.

Seed and preprocessing are exposed for inspection, **not per-run overrides**: the CLI has no flags for them. Edit `settings.py` only deliberately when no data job is running, rerun setup, and regenerate affected artifacts as needed; changing the displayed snapshot does not configure the CLI.

Export audits the prepared data and writes separate trainer and held-out-test ZIPs. With the current production defaults, the trainer archive is `models/guesser/data/.local/data/production/exports/trainer-production.zip`. The status cell prints resolved paths for your selections. Keep the held-out-test package sealed until the model and evaluation protocol are frozen.

In [ ]:
# Set only intentional overrides; None delegates defaults to the data CLI.
DATA_ROOT = None
PROFILE = DATA_DEFAULTS["profile"]
CONCURRENCY = None
CANDIDATE_RESERVE = None
CATEGORY_FILTER = ()
EXPORT_OUTPUT_DIR = None
INCLUDE_PREVIEW = False
RUN_PIPELINE = False

print(f"Data project: {DATA_PROJECT}")
print(f"Repository root: {REPOSITORY_ROOT}")

## Run the existing data commands

The small adapter below invokes the package's CLI with its locked Python 3.13 environment. It does not implement downloading, cleaning, splitting, auditing, or packaging. `subprocess.run(..., check=True)` streams command output into the notebook and stops the sequence on the first failure. Before each command it confirms the data package is installed in that environment; an environment that has never been synced stops the sequence with the `uv sync` command to run instead of a missing `guesser-data` script.

With `SYNC_DATA_ENVIRONMENT = False` and `RUN_PIPELINE = False`, running all notebook cells is safe: it skips dependency changes and production work, then reports status and the expected export path. Each stage prints a `[stage]` line here and then streams its own progress on stderr (`[download] 12/345 ...`, `[prepare] indexed 25/345 classes`, `[export] auditing train: ...`), so a long download, prepare, or export is visibly moving instead of looking hung. The stage result stays on stdout as JSON. Change either flag only when you deliberately want that operation and no data job is active.

> **Expected warning from the CLI calls below:** the message that `VIRTUAL_ENV=models/guesser/evaluation/.venv` does not match `models/guesser/data/.venv` and will be ignored is intentional. This kernel is Python 3.12, while `uv run --project models/guesser/data` selects the separate Python 3.13 data environment. Do not use `--active`; other errors or nonzero exits are still failures.

In [ ]:
def repository_path(value: str | Path) -> Path:
    """Resolve user paths with the same repository-root cwd as the data CLI."""
    return (REPOSITORY_ROOT / Path(value).expanduser()).resolve()


def run_data_command(command: str, *arguments: str) -> None:
    """Run one existing guesser-data command in its locked project environment."""
    # Let uv select the data project's Python 3.13 environment without syncing mid-run.
    command_line = [
        "uv",
        "run",
        "--project",
        str(DATA_PROJECT),
        "--frozen",
        "--no-sync",
        "guesser-data",
        command,
    ]
    # Pass explicit path/profile overrides while leaving other defaults to the CLI.
    if DATA_ROOT is not None:
        command_line.extend(["--data-root", str(repository_path(DATA_ROOT))])
    command_line.extend(["--profile", PROFILE, *arguments])
    # Refuse an unsynced data environment instead of uv's missing-script error.
    data_environment.require_data_environment(DATA_PROJECT, cwd=REPOSITORY_ROOT)
    # Mark the stage here; the CLI then streams its own progress lines live.
    print(f"[stage] {command} {' '.join(arguments)}".rstrip(), flush=True)
    # Inherit output so long-running downloads and actionable failures stay visible;
    # unbuffered Python keeps the progress lines arriving as the stage works.
    subprocess.run(
        command_line,
        cwd=REPOSITORY_ROOT,
        check=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )

In [ ]:
# Status is read-only; it shows whether the selected profile is ready for the next stage.
run_data_command("status")

# Report where the export step will write packages for the selected root and profile.
EFFECTIVE_DATA_ROOT = repository_path(
    DATA_DEFAULTS["data_root"] if DATA_ROOT is None else DATA_ROOT
)
EXPORT_DIRECTORY = (
    EFFECTIVE_DATA_ROOT / PROFILE / "exports"
    if EXPORT_OUTPUT_DIR is None
    else repository_path(EXPORT_OUTPUT_DIR)
)
print("Selected profile defaults:", DATA_DEFAULTS["profiles"][PROFILE])
print("Sampling method:", DATA_DEFAULTS["download_methods"][PROFILE])
print(f"Expected trainer package after export: {EXPORT_DIRECTORY / f'trainer-{PROFILE}.zip'}")
print(
    f"Expected held-out package after export: {EXPORT_DIRECTORY / f'held-out-test-{PROFILE}.zip'}"
)

In [ ]:
if not RUN_PIPELINE:
    # Keep opening or running all cells from starting expensive production work.
    print("Pipeline disabled. Set RUN_PIPELINE = True only when ready to run it.")
else:
    # Refuse example/probe profiles before any pipeline stage can write data.
    if PROFILE not in DATA_DEFAULTS["trainable_profiles"]:
        raise ValueError("Choose a trainable profile; smoke uses example/probe instead.")
    download_options = []
    if CONCURRENCY is not None:
        download_options.extend(["--concurrency", str(CONCURRENCY)])
    if CANDIDATE_RESERVE is not None:
        download_options.extend(["--candidates", str(CANDIDATE_RESERVE)])
    for category in CATEGORY_FILTER:
        download_options.extend(["--class", category])

    # Run the package's established stages in documented dependency order.
    run_data_command("download", *download_options)
    run_data_command("prepare")
    if INCLUDE_PREVIEW:
        run_data_command("preview")
    export_options = (
        []
        if EXPORT_OUTPUT_DIR is None
        else ["--output-dir", str(repository_path(EXPORT_OUTPUT_DIR))]
    )
    run_data_command("export", *export_options)
    run_data_command("status")

## Handoff

The trainer ZIP contains the training/validation package; the held-out-test ZIP is a distinct export and is not for tuning. Select the same profile in [`02_dataset_audit.ipynb`](02_dataset_audit.ipynb); it reads the matching expected counts from the data package. Its full split-isolation audit needs all three manifests, so a trainer-only ZIP is insufficient. The data CLI export audit checks payload integrity for the selected profile. Neither notebook evaluates recognition quality; that requires a frozen model checkpoint and predictions.